# Lab 08 — The profiler and the sweet-spot cost model (Objective O1 in miniature)

**Goal:** build the proposal's 'brain' end-to-end on one machine:
1. **measure** compute throughput T (matmul benchmark), packing cost L_pack(d), and link parameters (B, α — via a simulated link),
2. **predict** total step time per (d_fwd, d_bwd) configuration with the cost model,
3. **choose** the argmin under a convergence constraint (from Lab 05's cliff),
4. **validate** predicted vs measured — the plot that IS Objective O1,
5. run a mini **heterogeneity sweep** and watch the optimal depths change with the hardware.

In [ ]:
!pip -q install transformers matplotlib
import torch, time, itertools, matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
device = "cuda"
tok = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").to(device).eval()

def quantize(x, bits, block=64, stochastic=False):
    if bits >= 16: return x
    shape = x.shape; flat = x.float().flatten()
    pad = (-flat.numel()) % block
    if pad: flat = torch.cat([flat, flat.new_zeros(pad)])
    fb = flat.view(-1, block); qmax = 2**(bits-1)-1
    s = fb.abs().amax(1, keepdim=True)/qmax + 1e-12
    y = fb/s
    q = torch.floor(y + torch.rand_like(y)) if stochastic else torch.round(y)
    return (q.clamp(-qmax,qmax)*s).flatten()[:x.numel()].view(shape).to(x.dtype)

def cuda_time(fn, iters=20):
    s, e = torch.cuda.Event(True), torch.cuda.Event(True)
    fn()                                     # warmup
    torch.cuda.synchronize(); s.record()
    for _ in range(iters): fn()
    e.record(); torch.cuda.synchronize()
    return s.elapsed_time(e) / iters / 1000  # seconds

## 1. Profile the node — T (compute) and L_pack(d)
Never wall-clock CUDA: kernels are async. `cuda.Event` + `synchronize` or your numbers lie.

In [ ]:
A = torch.randn(2048, 2048, device=device, dtype=torch.float16)
t_mm = cuda_time(lambda: A @ A)
print(f"matmul throughput ~ {2*2048**3/t_mm/1e12:.1f} TFLOPS (this node's T)")

BSZ, S, H = 1, 512, 768                      # boundary tensor geometry
boundary = torch.randn(BSZ, S, H, device=device, dtype=torch.float16)
Lpack = {b: cuda_time(lambda b=b: quantize(boundary, b)) for b in [8, 6, 4]}
Lpack[16] = 0.0
for b, t in sorted(Lpack.items(), reverse=True):
    print(f"L_pack({b:2d}-bit) = {t*1e3:.3f} ms   (unoptimized PyTorch — Phase 3's Triton kernel attacks exactly this)")

## 2. A programmable link + measured stage compute

In [ ]:
class SimLink:
    def __init__(self, gbps, alpha_ms=0.2):
        self.B = gbps*1e9/8*0.94; self.alpha = alpha_ms/1e3
    def send(self, numel, bits):
        time.sleep(self.alpha + numel*bits/8/self.B)
    def wire_time(self, numel, bits):
        return self.alpha + numel*bits/8/self.B

ids = tok(" systems"*S, return_tensors="pt").input_ids[:, :S].to(device)
import torch.nn.functional as F
wte, wpe = model.transformer.wte, model.transformer.wpe
blocks, ln_f, head = model.transformer.h, model.transformer.ln_f, model.lm_head
CUT = 6
def stage1(x):
    h = wte(x) + wpe(torch.arange(x.size(1), device=device))
    for b in blocks[:CUT]: h = b(h)[0]
    return h
def stage2(h, x):
    for b in blocks[CUT:]: h = b(h)[0]
    lg = head(ln_f(h))
    return F.cross_entropy(lg[:, :-1].reshape(-1, lg.size(-1)), x[:, 1:].reshape(-1))

def t_compute():
    model.zero_grad()
    h = stage1(ids); r = h.detach().clone().requires_grad_()
    loss = stage2(r, ids); loss.backward(); h.backward(r.grad)
T_comp = cuda_time(t_compute, iters=5)
print(f"T_comp (fwd+bwd, both stages, no wire) = {T_comp*1e3:.1f} ms")

## 3. Predict, measure, choose
Cost model (sequential version — overlap is Phase 4's upgrade):
`T_step = T_comp + L_pack(d_f) + wire(S, d_f) + L_pack(d_b) + wire(S, d_b)`
Constraint from Lab 05's asymmetry curves: `d_b ≥ 6` (backward is fragile); `d_f ≥ 4`.

In [ ]:
def measured_step(link, bf, bb):
    model.zero_grad()
    t0 = time.perf_counter()
    h = stage1(ids)
    hw = quantize(h.detach(), bf); torch.cuda.synchronize()
    link.send(hw.numel(), bf)
    r = hw.clone().requires_grad_()
    loss = stage2(r, ids); loss.backward()
    gw = quantize(r.grad, bb, stochastic=True); torch.cuda.synchronize()
    link.send(gw.numel(), bb)
    h.backward(gw); torch.cuda.synchronize()
    return time.perf_counter() - t0

def predicted_step(link, bf, bb):
    n = BSZ*S*H
    return T_comp + Lpack[bf] + link.wire_time(n, bf) + Lpack[bb] + link.wire_time(n, bb)

link = SimLink(gbps=1.0)
configs = [(bf, bb) for bf, bb in itertools.product([16,8,6,4],[16,8,6]) if bf <= bb or True]
rows = []
for bf, bb in configs:
    if bf < 4 or bb < 6: continue
    p, m = predicted_step(link, bf, bb), measured_step(link, bf, bb)
    rows.append((bf, bb, p, m))
    print(f"d_fwd={bf:2d} d_bwd={bb:2d}   predicted {p*1e3:7.1f} ms   measured {m*1e3:7.1f} ms")
best = min(rows, key=lambda r: r[3])
print(f"\n>>> profiler's choice on this hardware/link: d_fwd={best[0]}, d_bwd={best[1]}")

In [ ]:
plt.figure(figsize=(4.5,4.5))
plt.scatter([r[2]*1e3 for r in rows], [r[3]*1e3 for r in rows])
lim = [0, max(r[3] for r in rows)*1e3*1.1]
plt.plot(lim, lim, "--", alpha=.5)
plt.xlabel("predicted step (ms)"); plt.ylabel("measured step (ms)")
plt.title("Cost-model validation — Objective O1's plot")
plt.grid(alpha=.3); plt.tight_layout(); plt.show()

Points hugging the diagonal = the cost model works. Report the mean % error; that number *is* O1's success criterion.

## 4. Mini heterogeneity sweep — watch the decision change
Same solver, different links: the optimal (d_fwd, d_bwd) should shift with bandwidth — uniform policies can't do this. That divergence is your thesis.

In [ ]:
print(f"{'link':>10s} {'best (df,db)':>14s} {'best ms':>9s} {'uniform-8 ms':>13s} {'fp16 ms':>9s} {'gain vs unif-8':>15s}")
for gbps in [0.1, 0.5, 1.0, 5.0, 10.0]:
    lk = SimLink(gbps)
    cand = [(bf, bb, predicted_step(lk, bf, bb)) for bf in [16,8,6,4] for bb in [16,8,6]]
    bf, bb, t = min(cand, key=lambda c: c[2])
    u8   = predicted_step(lk, 8, 8)
    fp16 = predicted_step(lk, 16, 16)
    print(f"{gbps:8.1f}G {f'({bf},{bb})':>14s} {t*1e3:8.1f} {u8*1e3:12.1f} {fp16*1e3:8.1f} {(u8/t-1)*100:13.1f}%")

Read the table: on fast links the profiler backs off compression (packing cost isn't worth it); on slow links it pushes to the constraint floor. A uniform policy is wrong at one end or the other — **the gap between columns grows with heterogeneity**, which is exactly the headline figure your final evaluation scales up.

## Exercises
1. Add `L_unpack` (time the dequantize alone) — how much does the prediction improve?
2. Sweep α (0.2ms → 50ms, the federated-WAN regime) at fixed bandwidth: when does latency dominate and make bit-depth irrelevant? (Your cost model's latency term, earning its keep.)
3. Emulate a *weak GPU*: multiply Lpack by 5 (a 3060 vs a 4090 has ~5x fewer SMs) and rerun the sweep — watch the profiler choose *lighter* compression on the weak node. That is SM-contention-awareness, demonstrated.
4. Replace SimLink with the real gloo wire from Lab 06 under netem, measure on two processes — the full Phase-2 profiler, real.